<!-- # Pipeline smoke test

Raw Citi Bike data -> `RawModelData` -> `ResolvedModelData` -> `Environment` -> `SimulationLog`.

The base scenario reproduces historical **demand** exactly. `FormDeparturesPhase`
re-releases every historical departure (gated by stock), and
`FormPotentialTripsPhase` assigns each departure a destination and duration from
the OD demand model `P(target | source, commodity)` + mean historical duration.

To make the replay exact, the resolved data is built with `saturate_stock=True`:
artificial saturated stock and dock capacities replace the GBFS snapshot, so the
demand gate and the overflow redirect stay in the pipeline but never bind. (The
GBFS snapshot is a current observation, unrelated to the historical start state,
and would starve the replay with stockouts that never happened.)

Because targets and durations come from the (aggregate) OD model rather than each
trip's own record, the per-trip journal is **not** identical to history, and the
OD-count matrix drifts slightly under per-period largest-remainder rounding. The
marginal that is preserved exactly is the departures table:
`simulated_departures_df == historical_departures_df`. -->

In [1]:
import pandas as pd

from gbp.loaders.dataloader_raw import RawModelData
from gbp.loaders.dataloader_graph import ResolvedModelData, attach_simulation
from gbp.consumers.simulator.engine import Environment, EnvironmentConfig
from gbp.consumers.simulator import (
    DockArrivals,
    FormDeparturesPhase,
    FormPotentialTripsPhase,
)

In [2]:
ubuntu_path = "/mnt/outer/Documents/vlzm/GFDRR_ubuntu/GFDRR/data/raw/202602-citibike-tripdata_1.csv"
mac_path = "/Users/vladislav/Documents/vlzm/GFDRR/data/raw/202601-citibike-tripdata_1.csv"

# Raw model data: read the trip CSV and the live GBFS feed, derive raw entities.
raw_data = RawModelData(
    gbfs_base="https://gbfs.citibikenyc.com/gbfs/en",
    trips_path=mac_path,
    seed=42,
    n_depots=10,
    depot_capacity=9000,
    n_trucks=5,
    truck_capacity_bikes=20,
    truck_rate=50.0,
    electric_bike_rate=5,
    classic_bike_rate=3,
)

graph_data = ResolvedModelData(raw_data, period_len=pd.Timedelta(hours=1), scale_capacity_factor = 10)

historical_flows_df_raw = graph_data.historical_flows_df.copy()

/Users/vladislav/Documents/vlzm/GFDRR/gbp/loaders/dataloader_raw.py:244: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.stations_capacities_df["capacity"] = 100


In [3]:
phases_canonical = [
    DockArrivals("previous"),
    FormDeparturesPhase(),
    FormPotentialTripsPhase(),
    DockArrivals("same"),
]

env_canonical = Environment(
    graph_data,
    EnvironmentConfig(phases=phases_canonical, 
                      seed=42, 
                      scenario_id="historical_replay", 
                      demand_scale_factor=1.0,
                      number_of_periods = 30),
)
env_canonical.run()

# Wire the finished run back into the graph-data container's simulated_* slots.
attach_simulation(graph_data, env_canonical.simulated_flows_df)
simulated_flows_df = graph_data.simulated_flows_df
simulated_departures_df = graph_data.simulated_departures_df
historical_departures_df = graph_data.historical_departures_df

In [4]:
from gbp.loaders.dataloader_graph import get_flows_wide, slice_flows_wide

simulated_flows_wild_df = get_flows_wide(graph_data)
simulated_flows_wild_df

,flow_id,move_id,event_id,period_id,flow_type,event_type,commodity_category,source_id,planned_target_id,realized_target_id,...,realized_target_capacity_total,realized_target_capacity_per_commodity_cat,realized_target_lat,realized_target_lng,realized_target_inventory_after,realized_target_inventory_before,planned_duration,realized_duration,planned_distance_km,realized_distance_km
0,sim_0_0,0,0,0,user_trip,departed,classic_bike,6626.01,5703.13,<NA>,...,NaN,NaN,NaN,NaN,<NA>,<NA>,17,<NA>,3.206291,NaN
1,sim_4_0,0,0,4,user_trip,departed,classic_bike,6224.06,6339.06,<NA>,...,NaN,NaN,NaN,NaN,<NA>,<NA>,24,<NA>,0.300107,NaN
2,sim_4_1,0,0,4,user_trip,departed,classic_bike,6257.06,6339.06,<NA>,...,NaN,NaN,NaN,NaN,<NA>,<NA>,24,<NA>,0.575615,NaN
3,sim_4_2,0,0,4,user_trip,departed,classic_bike,7599.09,7599.02,<NA>,...,NaN,NaN,NaN,NaN,<NA>,<NA>,25,<NA>,0.243180,NaN
4,sim_5_0,0,0,5,user_trip,departed,classic_bike,6030.04,6339.06,<NA>,...,NaN,NaN,NaN,NaN,<NA>,<NA>,23,<NA>,1.115145,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30217,sim_29_995,0,1,29,user_trip,arrived,classic_bike,6022.04,6030.04,6030.04,...,1000.0,1000.0,40.737815,-73.999947,15,15,0,0,0.639662,0.639662
30218,sim_29_996,0,1,29,user_trip,arrived,electric_bike,6022.04,6115.09,6115.09,...,1000.0,1000.0,40.742754,-74.007474,28,23,0,0,1.409936,1.409936
30219,sim_29_997,0,1,29,user_trip,arrived,electric_bike,6025.08,5542.04,5542.04,...,1000.0,1000.0,40.723250,-73.943080,20,21,0,0,2.129085,2.129085
30220,sim_29_998,0,1,29,user_trip,arrived,electric_bike,6025.08,6932.14,6932.14,...,1000.0,1000.0,40.767801,-73.965921,23,22,0,0,4.619503,4.619503


In [5]:
slice_flows_wide(simulated_flows_wild_df, "source", "6331.01", period_id=21, window=2)

,flow_id,move_id,event_id,period_id,flow_type,event_type,commodity_category,source_id,planned_target_id,realized_target_id,...,realized_target_capacity_total,realized_target_capacity_per_commodity_cat,realized_target_lat,realized_target_lng,realized_target_inventory_after,realized_target_inventory_before,planned_duration,realized_duration,planned_distance_km,realized_distance_km
11322,sim_18_662,0,1,19,user_trip,arrived,electric_bike,6331.01,6331.01,6331.01,...,1000.0,1000.0,40.749156,-73.991600,51,49,1,1,0.000000,0.000000
11618,sim_19_333,0,0,19,user_trip,departed,classic_bike,6331.01,6079.03,<NA>,...,NaN,NaN,NaN,NaN,<NA>,<NA>,1,<NA>,1.614602,NaN
12287,sim_19_333,0,1,20,user_trip,arrived,classic_bike,6331.01,6079.03,6079.03,...,1000.0,1000.0,40.741444,-73.975361,19,17,1,1,1.614602,1.614602
12971,sim_21_165,0,0,21,user_trip,departed,electric_bike,6331.01,6079.03,<NA>,...,NaN,NaN,NaN,NaN,<NA>,<NA>,0,<NA>,1.614602,NaN
12972,sim_21_166,0,0,21,user_trip,departed,electric_bike,6331.01,6089.11,<NA>,...,NaN,NaN,NaN,NaN,<NA>,<NA>,0,<NA>,1.262667,NaN
12973,sim_21_167,0,0,21,user_trip,departed,electric_bike,6331.01,6089.11,<NA>,...,NaN,NaN,NaN,NaN,<NA>,<NA>,0,<NA>,1.262667,NaN
13209,sim_21_165,0,1,21,user_trip,arrived,electric_bike,6331.01,6079.03,6079.03,...,1000.0,1000.0,40.741444,-73.975361,37,33,0,0,1.614602,1.614602
13210,sim_21_166,0,1,21,user_trip,arrived,electric_bike,6331.01,6089.11,6089.11,...,1000.0,1000.0,40.740693,-73.981606,27,25,0,0,1.262667,1.262667
13211,sim_21_167,0,1,21,user_trip,arrived,electric_bike,6331.01,6089.11,6089.11,...,1000.0,1000.0,40.740693,-73.981606,27,25,0,0,1.262667,1.262667
13489,sim_22_181,0,0,22,user_trip,departed,electric_bike,6331.01,6626.11,<NA>,...,NaN,NaN,NaN,NaN,<NA>,<NA>,0,<NA>,1.411290,NaN


In [6]:
slice_flows_wide(simulated_flows_wild_df, "planned_target",  "5294.04", period_id=12, window=2)

,flow_id,move_id,event_id,period_id,flow_type,event_type,commodity_category,source_id,planned_target_id,realized_target_id,...,realized_target_capacity_total,realized_target_capacity_per_commodity_cat,realized_target_lat,realized_target_lng,realized_target_inventory_after,realized_target_inventory_before,planned_duration,realized_duration,planned_distance_km,realized_distance_km


In [7]:
slice_flows_wide(simulated_flows_wild_df, "realized_target", "6233.04", period_id=20, window=0)

,flow_id,move_id,event_id,period_id,flow_type,event_type,commodity_category,source_id,planned_target_id,realized_target_id,...,realized_target_capacity_total,realized_target_capacity_per_commodity_cat,realized_target_lat,realized_target_lng,realized_target_inventory_after,realized_target_inventory_before,planned_duration,realized_duration,planned_distance_km,realized_distance_km


# Moment-level inventory and redirect inspection

The journal now carries `step_id` — the inventory time axis *below* the period.
A **step** is one `+1`/`-1` batch (dock arrivals from earlier periods, this
period's departures, same-period dock arrivals, and each redirect round); a
**moment** is the inventory just *before* or just *after* a step (Notations.md
§0.1). Three read-models sit on top of it:

- `inventory_at_moments` — every facility's inventory before/after each step (the
  fine view; per-period inventory is the value at each period's last step).
- `flows_with_inventory` — each event widened with its own facility's inventory
  before and after the event.
- `redirect_neighbor_table` — for one redirect, the full station's neighbours by
  distance out to the station the bike actually reached, each with its free docks
  at the redirect moment, so you can see *why* it landed where it did.

The base replay never overflows a dock, so it has **no** redirects. The first cell
shows the moment tools on the real run; the last two build a small nested-overflow
scenario (a bike skips a now-full nearer neighbour) to exercise
`redirect_neighbor_table`.

In [8]:
from gbp.model import (
    flows_with_inventory,
    get_inventory_df,
    inventory_at_moments,
    redirect_neighbor_table,
)

initial_inventory_df = graph_data.initial_inventory_df

# 1) The finalized journal now carries `step_id` -- the inventory time axis below
#    the period (one step per +1/-1 batch: dock-previous, departures, dock-same,
#    and each redirect round).
print("step_id range:",
      int(simulated_flows_df["step_id"].min()), "..", int(simulated_flows_df["step_id"].max()))
display(simulated_flows_df.head(8))

# 2) Inventory at every moment: each facility's value before/after each step.
moments_df = inventory_at_moments(simulated_flows_df, initial_inventory_df)
display(moments_df.head(8))

# 3) The journal widened with each event's own facility inventory before/after.
wide_flows_df = flows_with_inventory(simulated_flows_df, initial_inventory_df)
display(wide_flows_df.head(8))

# 4) Sanity: the fine view coarsens back to per-period inventory exactly
#    (a period's inventory is the value at its last step).
last_step = moments_df.groupby("period_id")["step_id"].transform("max") == moments_df["step_id"]
period_end_df = (moments_df[last_step]
                 .rename(columns={"inventory_after": "quantity"})
                 [["period_id", "facility_id", "commodity_category", "quantity"]])
coarse_df = get_inventory_df(simulated_flows_df, initial_inventory_df)
check_df = coarse_df.merge(period_end_df,
                           on=["period_id", "facility_id", "commodity_category"],
                           suffixes=("_coarse", "_fine"))
assert (check_df["quantity_coarse"] == check_df["quantity_fine"]).all()
print(f"OK: inventory_at_moments last step == get_inventory_df ({len(check_df)} rows match)")

step_id range: 0 .. 54


,flow_id,move_id,event_id,period_id,flow_type,event_type,commodity_category,source_id,planned_target_id,realized_target_id,start_period,planned_end_period,realized_end_period,resource_id,quantity,reason,phase_rank,redirect_round,step_id
0,sim_0_0,0,0,0,user_trip,departed,classic_bike,6626.01,5703.13,<NA>,0,17,<NA>,<NA>,1,<NA>,1,0,0
1,sim_4_0,0,0,4,user_trip,departed,classic_bike,6224.06,6339.06,<NA>,4,28,<NA>,<NA>,1,<NA>,1,0,1
2,sim_4_1,0,0,4,user_trip,departed,classic_bike,6257.06,6339.06,<NA>,4,28,<NA>,<NA>,1,<NA>,1,0,1
3,sim_4_2,0,0,4,user_trip,departed,classic_bike,7599.09,7599.02,<NA>,4,29,<NA>,<NA>,1,<NA>,1,0,1
4,sim_5_0,0,0,5,user_trip,departed,classic_bike,6030.04,6339.06,<NA>,5,28,<NA>,<NA>,1,<NA>,1,0,2
5,sim_5_1,0,0,5,user_trip,departed,classic_bike,6215.04,6339.06,<NA>,5,28,<NA>,<NA>,1,<NA>,1,0,2
6,sim_6_0,0,0,6,user_trip,departed,classic_bike,5679.05,6450.05,<NA>,6,27,<NA>,<NA>,1,<NA>,1,0,3
7,sim_6_1,0,0,6,user_trip,departed,classic_bike,6215.04,6450.05,<NA>,6,27,<NA>,<NA>,1,<NA>,1,0,3


,step_id,period_id,facility_id,commodity_category,inventory_before,inventory_after
0,0,0,1234.56,classic_bike,11,11
1,0,0,1234.56,electric_bike,14,14
2,0,0,1964.01,classic_bike,11,11
3,0,0,1964.01,electric_bike,13,13
4,0,0,2009.04,classic_bike,12,12
5,0,0,2009.04,electric_bike,11,11
6,0,0,2042.01,classic_bike,11,11
7,0,0,2042.01,electric_bike,11,11


,flow_id,move_id,event_id,period_id,flow_type,event_type,commodity_category,source_id,planned_target_id,realized_target_id,...,realized_end_period,resource_id,quantity,reason,phase_rank,redirect_round,step_id,facility_id,inventory_before,inventory_after
0,sim_0_0,0,0,0,user_trip,departed,classic_bike,6626.01,5703.13,<NA>,...,<NA>,<NA>,1,<NA>,1,0,0,6626.01,21,20
1,sim_4_0,0,0,4,user_trip,departed,classic_bike,6224.06,6339.06,<NA>,...,<NA>,<NA>,1,<NA>,1,0,1,6224.06,22,21
2,sim_4_1,0,0,4,user_trip,departed,classic_bike,6257.06,6339.06,<NA>,...,<NA>,<NA>,1,<NA>,1,0,1,6257.06,22,21
3,sim_4_2,0,0,4,user_trip,departed,classic_bike,7599.09,7599.02,<NA>,...,<NA>,<NA>,1,<NA>,1,0,1,7599.09,14,13
4,sim_5_0,0,0,5,user_trip,departed,classic_bike,6030.04,6339.06,<NA>,...,<NA>,<NA>,1,<NA>,1,0,2,6030.04,19,18
5,sim_5_1,0,0,5,user_trip,departed,classic_bike,6215.04,6339.06,<NA>,...,<NA>,<NA>,1,<NA>,1,0,2,6215.04,18,17
6,sim_6_0,0,0,6,user_trip,departed,classic_bike,5679.05,6450.05,<NA>,...,<NA>,<NA>,1,<NA>,1,0,3,5679.05,15,14
7,sim_6_1,0,0,6,user_trip,departed,classic_bike,6215.04,6450.05,<NA>,...,<NA>,<NA>,1,<NA>,1,0,3,6215.04,17,16


OK: inventory_at_moments last step == get_inventory_df (109750 rows match)


In [9]:
# The base replay never overflows a dock, so it has no redirects. Build a small
# scenario that does, with a nested overflow: five bikes aim at the full station
# s3; the nearest free neighbour s2 holds only two, then s4 holds two, then s5 --
# so the redirect spills across neighbours and later bikes skip a now-full one.
# (`tests` is not installed, so add the repo root to the path before importing.)
import pathlib
import sys

_repo_root = pathlib.Path.cwd()
while not (_repo_root / "pyproject.toml").exists() and _repo_root != _repo_root.parent:
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from tests import scenarios

# Filler trips in period 6 only exist to create the facilities s2/s4/s5.
_trips = [("s1", "s3", 0, 1)] * 5 + [("s1", "s2", 6, 7), ("s1", "s4", 6, 7), ("s1", "s5", 6, 7)]
redirect_resolved = scenarios.build_resolved(
    _trips, capacities={"s3": 0, "s2": 2, "s4": 2, "s5": 10}, initial_inventory={"s1": 8}
)
redirect_flows_df, _redirect_state = scenarios.run(redirect_resolved)

print("redirected events:", int((redirect_flows_df["event_type"] == "redirected").sum()))
display(flows_with_inventory(redirect_flows_df, redirect_resolved.initial_inventory_df))

redirected events: 7


/Users/vladislav/Documents/vlzm/GFDRR/gbp/consumers/simulator/state.py:210: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  flows = pd.concat([self.state_flows_df, new_flows], ignore_index=True)


,flow_id,move_id,event_id,period_id,flow_type,event_type,commodity_category,source_id,planned_target_id,realized_target_id,...,realized_end_period,resource_id,quantity,reason,phase_rank,redirect_round,step_id,facility_id,inventory_before,inventory_after
0,sim_0_0,0,0,0,user_trip,departed,classic_bike,s1,s3,<NA>,...,<NA>,<NA>,1,<NA>,1,0,0,s1,8,3
1,sim_0_1,0,0,0,user_trip,departed,classic_bike,s1,s3,<NA>,...,<NA>,<NA>,1,<NA>,1,0,0,s1,8,3
2,sim_0_2,0,0,0,user_trip,departed,classic_bike,s1,s3,<NA>,...,<NA>,<NA>,1,<NA>,1,0,0,s1,8,3
3,sim_0_3,0,0,0,user_trip,departed,classic_bike,s1,s3,<NA>,...,<NA>,<NA>,1,<NA>,1,0,0,s1,8,3
4,sim_0_4,0,0,0,user_trip,departed,classic_bike,s1,s3,<NA>,...,<NA>,<NA>,1,<NA>,1,0,0,s1,8,3
5,sim_0_0,0,1,1,user_trip,redirected,classic_bike,s1,s3,<NA>,...,1,<NA>,1,dock_full,0,1,1,<NA>,<NA>,<NA>
6,sim_0_0,1,2,1,user_trip,departed,classic_bike,s3,s2,<NA>,...,1,<NA>,1,<NA>,0,1,1,<NA>,<NA>,<NA>
7,sim_0_0,1,3,1,user_trip,arrived,classic_bike,s3,s2,s2,...,1,<NA>,1,<NA>,0,1,1,s2,0,2
8,sim_0_1,0,1,1,user_trip,redirected,classic_bike,s1,s3,<NA>,...,1,<NA>,1,dock_full,0,1,1,<NA>,<NA>,<NA>
9,sim_0_1,1,2,1,user_trip,departed,classic_bike,s3,s2,<NA>,...,1,<NA>,1,<NA>,0,1,1,<NA>,<NA>,<NA>


In [10]:
# Explain the redirect that docked at s4: by the time this bike was redirected,
# the nearer neighbour s2 was already full (free_before == 0), which is why it
# skipped s2 and landed at s4 (free_before >= 1). redirect_neighbor_table shows
# the full station's neighbours by distance out to where the bike actually docked,
# each with its free docks at the redirect moment -- the whole story in one table.
_dockings = flows_with_inventory(redirect_flows_df, redirect_resolved.initial_inventory_df)
_dockings = _dockings[(_dockings["move_id"] == 1) & (_dockings["event_type"] == "arrived")
                      & (_dockings["period_id"] == 1)]
example_flow_id = _dockings.loc[_dockings["realized_target_id"] == "s4", "flow_id"].iloc[0]
print("explaining redirect of flow:", example_flow_id, "(docked at s4)")

redirect_neighbor_table(
    redirect_flows_df,
    redirect_resolved.initial_inventory_df,
    redirect_resolved.facilities_geo_df,
    example_flow_id,
    capacities=redirect_resolved.facilities_capacities_df,
)

explaining redirect of flow: sim_0_2 (docked at s4)


,flow_id,step_id,period_id,planned_target_id,realized_target_id,commodity_category,neighbor_rank,facility_id,distance_sq,inventory_before,inventory_after,capacity,free_before,free_after
0,sim_0_2,2,1,s3,s4,classic_bike,0,s2,0.000002,2,2,2,0,0
1,sim_0_2,2,1,s3,s4,classic_bike,1,s4,0.000002,0,2,2,2,0
